# A6 hybrid LoRA — Colab A100

**Do not train on the Quadro.** Prefer **browser Colab** (especially if Cursor is on Remote SSH — Colab kernels often break there).

## Recommended: open in browser Colab

1. Open this link (signed into Google):
   [Open notebook in Colab](https://colab.research.google.com/github/IMJONEZZ/USU-LLM-Class-F26/blob/talha_shahzad_6/notebooks/colab_a100_hybrid_train.ipynb)
2. Runtime → Change runtime type → **A100** (Colab Pro) → Save.
3. Run cells top → bottom.

## Optional: Cursor Colab kernel (local Cursor only)

Only reliable if Cursor is **not** Remote-SSH’d to another machine:
1. Install Colab extension → `Colab: Sign In`.
2. Kernel picker → Colab → A100.
3. Cell 0 must print `Connected to Colab OK`.

## One Drive upload (once)

```bash
./scripts/pack_a6_for_colab.sh ./a6_prepared_hybrid.tgz
# Upload → Google Drive MyDrive/usu-a6/a6_prepared_hybrid.tgz
```

Need: HF token for `meta-llama/Llama-3.2-1B-Instruct`, branch `talha_shahzad_6` on GitHub.


In [ ]:
# 0) Must be a Colab kernel (from Cursor or browser)
import sys

IN_COLAB = "google.colab" in sys.modules or "COLAB_GPU" in __import__("os").environ
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    pass

assert IN_COLAB, (
    "This notebook is still on a LOCAL kernel.\n"
    "Kernel picker → Select Another Kernel → Colab → A100, then re-run."
)
print("Connected to Colab OK")

In [ ]:
# 1) GPU check (expect A100)
import subprocess

import torch

print(subprocess.getoutput("nvidia-smi"))
assert torch.cuda.is_available(), "No CUDA — pick a Colab GPU runtime"
name = torch.cuda.get_device_name(0)
vram = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1)
print(f"GPU={name} VRAM_GiB={vram}")
if "A100" not in name and "H100" not in name:
    print(
        "WARNING: not A100/H100 — training will be slower. Change Colab runtime if you can."
    )

In [ ]:
# 2) Mount Drive, clone repo, unpack prepared hybrid data
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

# Change only if your Drive path differs
DRIVE_TGZ = "/content/drive/MyDrive/usu-a6/a6_prepared_hybrid.tgz"
REPO_URL = "https://github.com/IMJONEZZ/USU-LLM-Class-F26.git"
BRANCH = "talha_shahzad_6"
REPO = Path("/content/USU-LLM-Class-F26")

assert Path(DRIVE_TGZ).is_file(), (
    f"Missing {DRIVE_TGZ}. Pack on laptop with ./scripts/pack_a6_for_colab.sh and upload to Drive."
)

import shutil
import subprocess

if REPO.exists():
    shutil.rmtree(REPO)
subprocess.check_call(
    ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(REPO)]
)
subprocess.check_call(["tar", "-xzf", DRIVE_TGZ, "-C", str(REPO)])

%cd /content/USU-LLM-Class-F26
!ls -la data/assignment6/prepared_hybrid data/assignment6/qa/hybrid_train.jsonl

In [ ]:
# 3) Install Unsloth + train deps on the Colab VM
%pip -q install -U pip
%pip -q install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip -q install peft trl accelerate bitsandbytes datasets "transformers==4.57.6"
%pip -q install evaluate rouge-score bert-score

In [ ]:
# 4) Hugging Face login (gated Llama Instruct)
import getpass
import os

from huggingface_hub import login

token = os.environ.get("HF_TOKEN") or getpass.getpass("HF token: ")
login(token=token)
os.environ["HF_TOKEN"] = token

In [ ]:
# 5) Feasibility + train on A100 with large micro-batches
%cd /content/USU-LLM-Class-F26
from pathlib import Path
import subprocess

BATCH_SIZE = 128
ACCUMULATION = 1
EPOCHS = 3
FEASIBILITY_STEPS = 4

!rm -rf data/assignment6/feasibility_hybrid_colab data/assignment6/train_hybrid_colab

common = [
    "--prepared", "data/assignment6/prepared_hybrid",
    "--batch-size", str(BATCH_SIZE),
    "--accumulation", str(ACCUMULATION),
    "--learning-rate", "2e-4",
    "--rank", "4",
    "--alpha", "8",
    "--targets", "q_proj,v_proj",
    "--no-gradient-checkpointing",
]
subprocess.check_call([
    "python", "-m", "src.trainer", "feasibility", *common,
    "--output", "data/assignment6/feasibility_hybrid_colab",
    "--steps", str(FEASIBILITY_STEPS),
])
subprocess.check_call([
    "python", "-m", "src.trainer", "train", *common,
    "--feasibility", "data/assignment6/feasibility_hybrid_colab",
    "--output", "data/assignment6/train_hybrid_colab",
    "--epochs", str(EPOCHS),
])
assert Path("data/assignment6/train_hybrid_colab/success.json").is_file()
print("TRAIN_OK")
!nvidia-smi
!ls -la data/assignment6/train_hybrid_colab/best


In [ ]:
# 6) Save adapter to Drive (download later to laptop)
import json
import shutil
from pathlib import Path

src = Path("data/assignment6/train_hybrid_colab/best")
dst = Path("/content/drive/MyDrive/usu-a6/adapters/hybrid_colab_a100")
dst.parent.mkdir(parents=True, exist_ok=True)
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(src, dst)
shutil.copy("data/assignment6/train_hybrid_colab/training.json", dst / "training.json")
shutil.copy("data/assignment6/train_hybrid_colab/success.json", dst / "success.json")
meta = json.loads((dst / "run.json").read_text()) if (dst / "run.json").exists() else {}
print("Saved:", dst)
print(
    "epoch", meta.get("checkpoint_epoch"), "dev_loss", meta.get("checkpoint_dev_loss")
)
print(
    "On laptop: copy Drive folder → data/assignment6/adapters/hybrid_colab_a100 "
    "then re-run graded-10 eval."
)